# Evaluating Model Outputs

We can evaluate a model's confidence in its results by using perplexity. Perplexity is a measure of uncertainty that can be calculated by exponentiating the negative of the average of the logprobs. 

+ Perplexity can be used to assess the result of an individual model run.
+ It can also be used to compare the relative confidence of results between model runs. 

Low perplexity or high confidence does not guarantee accuracy, but it can be a helpful signal when paired with other evaluation metrics. 

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [2]:
from utils.clients import get_client
from IPython.display import display, Markdown
import numpy as np

client = get_client()

In [3]:
prompts = [
    # Low perplexity: Clear topic, common structure, highly predictable vocabulary
    "Explain how photosynthesis works in simple terms.",
    # Medium preplexity: Narrative freedom, but familiar theme and constraints.
    "Write a short story about a traveler who realizes the journey mattered more than the destination.",
    # High perplexity: Abstract concept, creative freedom, unpredictable vocabulary
    "Describe the taste of a color that only exists for one second at dusk, using metaphors from mathematics and weather."
]

In [5]:
def get_completion(
    input: list[dict[str, str]],
    model: str = "gpt-4o-mini",
    max_tokens=500,
    temperature=0,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "input": input,
        "max_output_tokens": max_tokens,
        "temperature": temperature,
        "tools": tools,
        "include": ["message.output_text.logprobs"] if logprobs else [],
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.responses.create(**params)
    return completion

In [6]:
for k, prompt in enumerate(prompts):
    API_RESPONSE = get_completion(
        [{"role": "user", "content": prompt}],
        model="gpt-4o-mini",
        logprobs=True,
    )
    token_data = API_RESPONSE.output[0].content[0].logprobs
    response_text = API_RESPONSE.output[0].content[0].text
    lp_values = [t.logprob for t in token_data]
    perplexity_score = np.exp(-np.mean(lp_values))

    rows = [
        f"| `{t.token.replace('|', chr(9474)).replace(chr(10), '↵').replace(chr(96), chr(39))}` "
        f"| {t.logprob:.4f} | {np.exp(t.logprob)*100:.1f}% |"
        for t in token_data
    ]
    table = "\n".join([
        "| Token | Logprob | Linear Prob |",
        "|:------|--------:|------------:|",
    ] + rows)

    display(Markdown(f"### Prompt {k+1}\n_{prompt}_"))
    display(Markdown(f"**Response:**\n\n{response_text}"))
    display(Markdown(table))
    display(Markdown(f"**Perplexity:** `{perplexity_score:.2f}`\n\n---"))

### Prompt 1
_Explain how photosynthesis works in simple terms._

**Response:**

Photosynthesis is the process that plants, algae, and some bacteria use to make their own food. Here’s how it works in simple terms:

1. **Sunlight**: Plants take in sunlight using a green pigment called chlorophyll, which is found in their leaves.

2. **Water**: They absorb water from the soil through their roots.

3. **Carbon Dioxide**: Plants take in carbon dioxide from the air through tiny openings in their leaves called stomata.

4. **Making Food**: Using the energy from sunlight, plants combine water and carbon dioxide to create glucose (a type of sugar) and oxygen. The glucose is used as food for energy and growth.

5. **Oxygen Release**: The oxygen produced during this process is released into the air, which is essential for us and other living beings to breathe.

In summary, photosynthesis is how plants turn sunlight, water, and carbon dioxide into food and oxygen!

| Token | Logprob | Linear Prob |
|:------|--------:|------------:|
| `Photos` | -0.0793 | 92.4% |
| `ynthesis` | 0.0000 | 100.0% |
| ` is` | 0.0000 | 100.0% |
| ` the` | -0.0436 | 95.7% |
| ` process` | -0.0041 | 99.6% |
| ` that` | -0.2283 | 79.6% |
| ` plants` | -0.0049 | 99.5% |
| `,` | -0.5768 | 56.2% |
| ` algae` | -0.0296 | 97.1% |
| `,` | 0.0000 | 100.0% |
| ` and` | 0.0000 | 100.0% |
| ` some` | -0.0000 | 100.0% |
| ` bacteria` | -0.0001 | 100.0% |
| ` use` | -0.0000 | 100.0% |
| ` to` | 0.0000 | 100.0% |
| ` make` | -0.1557 | 85.6% |
| ` their` | -0.0067 | 99.3% |
| ` own` | -0.0789 | 92.4% |
| ` food` | -0.0000 | 100.0% |
| `.` | -0.5787 | 56.1% |
| ` Here` | -0.2042 | 81.5% |
| `’s` | -0.0001 | 100.0% |
| ` how` | -0.1002 | 90.5% |
| ` it` | 0.0000 | 100.0% |
| ` works` | -0.0000 | 100.0% |
| ` in` | -0.0238 | 97.6% |
| ` simple` | -0.0001 | 100.0% |
| ` terms` | -0.0015 | 99.8% |
| `:↵↵` | -0.0000 | 100.0% |
| `1` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Sun` | -0.7148 | 48.9% |
| `light` | -0.0003 | 100.0% |
| `**` | -0.0996 | 90.5% |
| `:` | -0.0000 | 100.0% |
| ` Plants` | -0.0010 | 99.9% |
| ` take` | -0.8876 | 41.2% |
| ` in` | -0.0007 | 99.9% |
| ` sunlight` | -0.0190 | 98.1% |
| ` using` | -0.5051 | 60.3% |
| ` a` | -0.1045 | 90.1% |
| ` green` | -0.0336 | 96.7% |
| ` pigment` | -0.0021 | 99.8% |
| ` called` | -0.0486 | 95.3% |
| ` chlor` | -0.0000 | 100.0% |
| `ophyll` | 0.0000 | 100.0% |
| `,` | -0.5955 | 55.1% |
| ` which` | -0.6941 | 50.0% |
| ` is` | -0.0001 | 100.0% |
| ` found` | -0.3859 | 68.0% |
| ` in` | -0.0130 | 98.7% |
| ` their` | -0.0025 | 99.8% |
| ` leaves` | -0.0000 | 100.0% |
| `.↵↵` | -0.0030 | 99.7% |
| `2` | 0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Water` | -0.3352 | 71.5% |
| `**` | -0.3154 | 73.0% |
| `:` | 0.0000 | 100.0% |
| ` They` | -0.6425 | 52.6% |
| ` absorb` | -0.0794 | 92.4% |
| ` water` | -0.0000 | 100.0% |
| ` from` | -0.0843 | 91.9% |
| ` the` | -0.0001 | 100.0% |
| ` soil` | -0.0182 | 98.2% |
| ` through` | -0.0000 | 100.0% |
| ` their` | -0.0000 | 100.0% |
| ` roots` | -0.0000 | 100.0% |
| `.↵↵` | -0.0005 | 100.0% |
| `3` | 0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Carbon` | -0.0004 | 100.0% |
| ` D` | -0.0052 | 99.5% |
| `ioxide` | -0.0000 | 100.0% |
| `**` | -0.0002 | 100.0% |
| `:` | 0.0000 | 100.0% |
| ` Plants` | -0.0100 | 99.0% |
| ` take` | -0.3516 | 70.4% |
| ` in` | -0.0006 | 99.9% |
| ` carbon` | -0.0041 | 99.6% |
| ` dioxide` | 0.0000 | 100.0% |
| ` from` | -0.0470 | 95.4% |
| ` the` | -0.0000 | 100.0% |
| ` air` | -0.0000 | 100.0% |
| ` through` | -0.0002 | 100.0% |
| ` tiny` | -0.0790 | 92.4% |
| ` openings` | -0.0113 | 98.9% |
| ` in` | -0.0190 | 98.1% |
| ` their` | -0.0004 | 100.0% |
| ` leaves` | 0.0000 | 100.0% |
| ` called` | -0.0066 | 99.3% |
| ` stom` | -0.0002 | 100.0% |
| `ata` | -0.0000 | 100.0% |
| `.↵↵` | -0.0000 | 100.0% |
| `4` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Making` | -1.0409 | 35.3% |
| ` Food` | -0.0028 | 99.7% |
| `**` | -0.0002 | 100.0% |
| `:` | -0.0001 | 100.0% |
| ` Using` | -0.0367 | 96.4% |
| ` the` | -0.4079 | 66.5% |
| ` energy` | -0.2529 | 77.7% |
| ` from` | -0.0000 | 100.0% |
| ` sunlight` | -0.0041 | 99.6% |
| `,` | -0.0000 | 100.0% |
| ` plants` | -0.0091 | 99.1% |
| ` combine` | -0.2068 | 81.3% |
| ` water` | -0.1632 | 84.9% |
| ` and` | -0.0001 | 100.0% |
| ` carbon` | 0.0000 | 100.0% |
| ` dioxide` | 0.0000 | 100.0% |
| ` to` | -0.0043 | 99.6% |
| ` create` | -0.2327 | 79.2% |
| ` glucose` | -0.0205 | 98.0% |
| ` (` | -0.0488 | 95.2% |
| `a` | -0.0003 | 100.0% |
| ` type` | -0.0120 | 98.8% |
| ` of` | 0.0000 | 100.0% |
| ` sugar` | -0.0000 | 100.0% |
| `)` | -0.0879 | 91.6% |
| ` and` | -0.0016 | 99.8% |
| ` oxygen` | -0.0004 | 100.0% |
| `.` | -0.2017 | 81.7% |
| ` The` | -0.6304 | 53.2% |
| ` glucose` | -0.3216 | 72.5% |
| ` is` | -0.5949 | 55.2% |
| ` used` | -0.4933 | 61.1% |
| ` as` | -0.2463 | 78.2% |
| ` food` | -0.0234 | 97.7% |
| ` for` | -0.0581 | 94.4% |
| ` energy` | -0.4329 | 64.9% |
| ` and` | -0.0147 | 98.5% |
| ` growth` | -0.0000 | 100.0% |
| `.↵↵` | -0.2524 | 77.7% |
| `5` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `O` | -0.1520 | 85.9% |
| `xygen` | -0.0000 | 100.0% |
| ` Release` | -0.0410 | 96.0% |
| `**` | -0.0000 | 100.0% |
| `:` | 0.0000 | 100.0% |
| ` The` | -0.5046 | 60.4% |
| ` oxygen` | -0.1848 | 83.1% |
| ` produced` | -0.0260 | 97.4% |
| ` during` | -0.1534 | 85.8% |
| ` this` | -0.0209 | 97.9% |
| ` process` | -0.0000 | 100.0% |
| ` is` | -0.0002 | 100.0% |
| ` released` | -0.0030 | 99.7% |
| ` into` | -0.6932 | 50.0% |
| ` the` | 0.0000 | 100.0% |
| ` air` | -0.0086 | 99.1% |
| `,` | -0.0114 | 98.9% |
| ` which` | -0.0014 | 99.9% |
| ` is` | -0.0325 | 96.8% |
| ` essential` | -1.1533 | 31.6% |
| ` for` | -0.0000 | 100.0% |
| ` us` | -0.8738 | 41.7% |
| ` and` | -0.1297 | 87.8% |
| ` other` | -0.0499 | 95.1% |
| ` living` | -0.1029 | 90.2% |
| ` beings` | -0.4070 | 66.6% |
| ` to` | -0.0183 | 98.2% |
| ` breathe` | -0.0000 | 100.0% |
| `.↵↵` | -0.0006 | 99.9% |
| `In` | -0.4751 | 62.2% |
| ` summary` | -0.2070 | 81.3% |
| `,` | -0.0068 | 99.3% |
| ` photos` | -0.1057 | 90.0% |
| `ynthesis` | 0.0000 | 100.0% |
| ` is` | -1.0611 | 34.6% |
| ` how` | -0.0818 | 92.1% |
| ` plants` | -0.0001 | 100.0% |
| ` turn` | -0.3235 | 72.4% |
| ` sunlight` | -0.0068 | 99.3% |
| `,` | -0.0232 | 97.7% |
| ` water` | -0.0001 | 100.0% |
| `,` | -0.0000 | 100.0% |
| ` and` | 0.0000 | 100.0% |
| ` carbon` | -0.0182 | 98.2% |
| ` dioxide` | 0.0000 | 100.0% |
| ` into` | 0.0000 | 100.0% |
| ` food` | -0.0052 | 99.5% |
| ` and` | -0.1623 | 85.0% |
| ` oxygen` | -0.0016 | 99.8% |
| `!` | -0.2150 | 80.7% |

**Perplexity:** `1.12`

---

### Prompt 2
_Write a short story about a traveler who realizes the journey mattered more than the destination._

...

**Perplexity:** `1.52`

---

### Prompt 3
_Describe the taste of a color that only exists for one second at dusk, using metaphors from mathematics and weather._

**Response:**

Imagine a color that tastes like the fleeting moment when twilight kisses the horizon—a blend of soft lavender and deep indigo, like the gentle curve of a parabolic arc just before it meets the ground. It’s the taste of a cool breeze, whispering secrets of impending rain, where each drop is a note in a symphony of flavors.

This color is a fleeting equation, a perfect balance of sweet and tart, like the intersection of two lines that only meet for an instant. It’s the tang of a summer storm, where the air is thick with anticipation, and the taste is both refreshing and electric, like the sudden spark of lightning illuminating the sky.

As it fades, it leaves a lingering sensation, akin to the gentle slope of a sine wave, rising and falling in a dance of ephemeral beauty. It’s a momentary taste of infinity, where time stretches and contracts, leaving you with the bittersweet aftertaste of a sunset that was never meant to last.

| Token | Logprob | Linear Prob |
|:------|--------:|------------:|
| `Imagine` | -0.3913 | 67.6% |
| ` a` | -0.5278 | 59.0% |
| ` color` | -0.4882 | 61.4% |
| ` that` | -0.0104 | 99.0% |
| ` tastes` | -1.4708 | 23.0% |
| ` like` | -0.0003 | 100.0% |
| ` the` | -0.2201 | 80.2% |
| ` fleeting` | -0.1942 | 82.4% |
| ` moment` | -0.2669 | 76.6% |
| ` when` | -0.2223 | 80.1% |
| ` twilight` | -0.7535 | 47.1% |
| ` kisses` | -1.8835 | 15.2% |
| ` the` | -0.0451 | 95.6% |
| ` horizon` | -0.0915 | 91.3% |
| `—a` | -0.7702 | 46.3% |
| ` blend` | -1.6874 | 18.5% |
| ` of` | -0.0232 | 97.7% |
| ` soft` | -1.6409 | 19.4% |
| ` lavender` | -0.4658 | 62.8% |
| ` and` | -0.0123 | 98.8% |
| ` deep` | -0.3122 | 73.2% |
| ` ind` | -0.1336 | 87.5% |
| `igo` | -0.0000 | 100.0% |
| `,` | -0.3608 | 69.7% |
| ` like` | -1.0721 | 34.2% |
| ` the` | -0.4072 | 66.6% |
| ` gentle` | -0.5727 | 56.4% |
| ` curve` | -0.3722 | 68.9% |
| ` of` | -0.0000 | 100.0% |
| ` a` | -0.0263 | 97.4% |
| ` par` | -0.4370 | 64.6% |
| `abolic` | -0.0013 | 99.9% |
| ` arc` | -0.0492 | 95.2% |
| ` just` | -1.2122 | 29.8% |
| ` before` | -0.0131 | 98.7% |
| ` it` | -0.0167 | 98.3% |
| ` meets` | -1.5304 | 21.6% |
| ` the` | -0.2237 | 80.0% |
| ` ground` | -0.5586 | 57.2% |
| `.` | -0.0397 | 96.1% |
| ` It` | -0.8291 | 43.6% |
| `’s` | -0.4347 | 64.7% |
| ` the` | -0.6357 | 53.0% |
| ` taste` | -1.4783 | 22.8% |
| ` of` | -0.0003 | 100.0% |
| ` a` | -0.2939 | 74.5% |
| ` cool` | -0.7987 | 45.0% |
| ` breeze` | -0.0456 | 95.5% |
| `,` | -1.3337 | 26.3% |
| ` whisper` | -1.3495 | 25.9% |
| `ing` | -0.0001 | 100.0% |
| ` secrets` | -0.6362 | 52.9% |
| ` of` | -0.1819 | 83.4% |
| ` impending` | -1.7069 | 18.1% |
| ` rain` | -0.4974 | 60.8% |
| `,` | -0.0560 | 94.5% |
| ` where` | -1.5656 | 20.9% |
| ` each` | -0.4785 | 62.0% |
| ` drop` | -0.4601 | 63.1% |
| ` is` | -0.6355 | 53.0% |
| ` a` | -0.0617 | 94.0% |
| ` note` | -1.6715 | 18.8% |
| ` in` | -0.0354 | 96.5% |
| ` a` | -0.1635 | 84.9% |
| ` sym` | -0.5949 | 55.2% |
| `phony` | -0.0549 | 94.7% |
| ` of` | -0.0702 | 93.2% |
| ` flavors` | -2.3019 | 10.0% |
| `.↵↵` | -0.9496 | 38.7% |
| `This` | -0.1508 | 86.0% |
| ` color` | -0.3407 | 71.1% |
| ` is` | -0.9235 | 39.7% |
| ` a` | -0.9555 | 38.5% |
| ` fleeting` | -0.7149 | 48.9% |
| ` equation` | -1.0230 | 36.0% |
| `,` | -0.0813 | 92.2% |
| ` a` | -1.0316 | 35.6% |
| ` perfect` | -1.8620 | 15.5% |
| ` balance` | -0.3116 | 73.2% |
| ` of` | -0.6707 | 51.1% |
| ` sweet` | -0.3134 | 73.1% |
| ` and` | -0.0077 | 99.2% |
| ` tart` | -0.9109 | 40.2% |
| `,` | -0.0858 | 91.8% |
| ` like` | -0.9967 | 36.9% |
| ` the` | -0.1240 | 88.3% |
| ` intersection` | -0.7881 | 45.5% |
| ` of` | -0.0517 | 95.0% |
| ` two` | -0.1323 | 87.6% |
| ` lines` | -0.0966 | 90.8% |
| ` that` | -0.9416 | 39.0% |
| ` only` | -1.0981 | 33.4% |
| ` meet` | -0.8727 | 41.8% |
| ` for` | -0.4772 | 62.1% |
| ` an` | -0.4327 | 64.9% |
| ` instant` | -0.0317 | 96.9% |
| `.` | -0.7121 | 49.1% |
| ` It` | -0.1371 | 87.2% |
| `’s` | -0.8809 | 41.4% |
| ` the` | -0.4287 | 65.1% |
| ` tang` | -1.7535 | 17.3% |
| ` of` | -0.0183 | 98.2% |
| ` a` | -1.4497 | 23.5% |
| ` summer` | -1.4136 | 24.3% |
| ` storm` | -0.0906 | 91.3% |
| `,` | -0.4020 | 66.9% |
| ` where` | -1.6211 | 19.8% |
| ` the` | -0.3495 | 70.5% |
| ` air` | -0.5128 | 59.9% |
| ` is` | -0.4497 | 63.8% |
| ` thick` | -0.3481 | 70.6% |
| ` with` | -0.0307 | 97.0% |
| ` anticipation` | -0.9640 | 38.1% |
| `,` | -0.1343 | 87.4% |
| ` and` | -0.7818 | 45.8% |
| ` the` | -0.4429 | 64.2% |
| ` taste` | -1.3023 | 27.2% |
| ` is` | -0.6793 | 50.7% |
| ` both` | -0.9555 | 38.5% |
| ` refreshing` | -0.9171 | 40.0% |
| ` and` | -0.0020 | 99.8% |
| ` electric` | -0.1231 | 88.4% |
| `,` | -0.4435 | 64.2% |
| ` like` | -0.8448 | 43.0% |
| ` the` | -0.5817 | 55.9% |
| ` sudden` | -1.4225 | 24.1% |
| ` spark` | -1.9931 | 13.6% |
| ` of` | -0.0174 | 98.3% |
| ` lightning` | -0.5365 | 58.5% |
| ` illuminating` | -0.3085 | 73.5% |
| ` the` | -0.6703 | 51.2% |
| ` sky` | -0.7779 | 45.9% |
| `.↵↵` | -0.2329 | 79.2% |
| `As` | -0.4839 | 61.6% |
| ` it` | -0.4883 | 61.4% |
| ` fades` | -1.1052 | 33.1% |
| `,` | -0.2619 | 77.0% |
| ` it` | -0.3053 | 73.7% |
| ` leaves` | -0.1598 | 85.2% |
| ` a` | -0.7059 | 49.4% |
| ` lingering` | -1.1899 | 30.4% |
| ` sensation` | -1.2443 | 28.8% |
| `,` | -0.5588 | 57.2% |
| ` akin` | -0.8250 | 43.8% |
| ` to` | 0.0000 | 100.0% |
| ` the` | -0.0449 | 95.6% |
| ` gentle` | -1.5294 | 21.7% |
| ` slope` | -0.5117 | 59.9% |
| ` of` | -0.0023 | 99.8% |
| ` a` | -0.0245 | 97.6% |
| ` sine` | -0.2905 | 74.8% |
| ` wave` | -0.0145 | 98.6% |
| `,` | -0.3970 | 67.2% |
| ` rising` | -0.7976 | 45.0% |
| ` and` | -0.0198 | 98.0% |
| ` falling` | -0.0031 | 99.7% |
| ` in` | -1.0353 | 35.5% |
| ` a` | -1.5957 | 20.3% |
| ` dance` | -1.0534 | 34.9% |
| ` of` | -0.3314 | 71.8% |
| ` ephemeral` | -1.3790 | 25.2% |
| ` beauty` | -0.2646 | 76.8% |
| `.` | -0.7190 | 48.7% |
| ` It` | -1.0408 | 35.3% |
| `’s` | -0.1388 | 87.0% |
| ` a` | -0.5426 | 58.1% |
| ` moment` | -0.7997 | 44.9% |
| `ary` | -1.1794 | 30.7% |
| ` taste` | -1.1774 | 30.8% |
| ` of` | -0.2490 | 78.0% |
| ` infinity` | -1.1088 | 33.0% |
| `,` | -0.1410 | 86.9% |
| ` where` | -1.3123 | 26.9% |
| ` time` | -0.6954 | 49.9% |
| ` stretches` | -1.1178 | 32.7% |
| ` and` | -0.4273 | 65.2% |
| ` contracts` | -0.1797 | 83.6% |
| `,` | -0.1871 | 82.9% |
| ` leaving` | -1.4766 | 22.8% |
| ` you` | -0.6651 | 51.4% |
| ` with` | -1.1237 | 32.5% |
| ` the` | -0.4957 | 60.9% |
| ` bitters` | -1.2397 | 28.9% |
| `weet` | 0.0000 | 100.0% |
| ` after` | -0.3438 | 70.9% |
| `taste` | -0.1620 | 85.0% |
| ` of` | -0.0010 | 99.9% |
| ` a` | -0.9585 | 38.3% |
| ` sunset` | -1.0310 | 35.7% |
| ` that` | -0.6539 | 52.0% |
| ` was` | -1.6245 | 19.7% |
| ` never` | -0.9053 | 40.4% |
| ` meant` | -0.0315 | 96.9% |
| ` to` | -0.0000 | 100.0% |
| ` last` | -0.0776 | 92.5% |
| `.` | -0.0283 | 97.2% |

**Perplexity:** `1.81`

---